In [39]:
import pandas as pd
import numpy as np
import re
pd.set_option('display.max_columns', None)

In [69]:
df = pd.read_csv("../../datasets/raw/steam_games_requirements.csv")

In [41]:
df.head(1)

,Unnamed: 0,url,types,name,desc_snippet,recent_reviews,all_reviews,release_date,developer,publisher,popular_tags,game_details,languages,achievements,genre,game_description,mature_content,minimum_requirements,recommended_requirements,original_price,discount_price
0,0,https://store.steampowered.com/app/379720/DOOM/,app,DOOM,Now includes all three premium DLC packs (Unto...,"Very Positive,(554),- 89% of the 554 user revi...","Very Positive,(42,550),- 92% of the 42,550 use...","May 12, 2016",id Software,"Bethesda Softworks,Bethesda Softworks","FPS,Gore,Action,Demons,Shooter,First-Person,Gr...","Single-player,Multi-player,Co-op,Steam Achieve...","English,French,Italian,German,Spanish - Spain,...",54.0,Action,"About This Game Developed by id software, the...",NaN,"Minimum:,OS:,Windows 7/8.1/10 (64-bit versions...","Recommended:,OS:,Windows 7/8.1/10 (64-bit vers...",$19.99,$14.99


In [42]:
df = df[df['types'] == 'app']
df.drop(columns=['types'], inplace=True)

In [43]:
df['url'] = df['url'].apply(lambda row: row.split('/')[4])

In [44]:
df.rename(columns={'url': 'app_id'}, inplace=True)

In [45]:
df = df[['app_id', 'name', 'minimum_requirements', 'recommended_requirements']]

In [46]:
df.head(1)

,app_id,name,minimum_requirements,recommended_requirements
0,379720,DOOM,"Minimum:,OS:,Windows 7/8.1/10 (64-bit versions...","Recommended:,OS:,Windows 7/8.1/10 (64-bit vers..."


In [47]:
df.isna().sum()

app_id                          0
name                           14
minimum_requirements        16952
recommended_requirements    16946
dtype: int64

In [48]:
df = df.dropna()

In [49]:
def extract_cpu(strings):
    if pd.isna(strings):
        return None
    if 'Processor:' in strings:
        s = strings.split(',')
        return s[s.index('Processor:')+1]
    return None

In [50]:
req = df[df['minimum_requirements'].str.contains('Processor|CPU', case=True)]['minimum_requirements'].tolist()

In [51]:
def extract_cpu2(text):
    """Extract CPU info from Processor line."""
    CPU_LINE_PATTERN = r'(?i)(?:Processor|CPU):[,\s]+([^,]+)'
    if pd.isna(text):
        return None
    text = str(text)
    match = re.search(CPU_LINE_PATTERN, text)
    if match:
        return match.group(1).strip()
    return None

In [52]:
extract_cpu2(req[110])

'Core i5-760 or better / AMD Phenom II X4 or better [Quad-core CPU]'

In [62]:
df['rec_cpu'] = df['recommended_requirements'].apply(lambda x: extract_cpu2(x))

In [67]:
df['min_cpu'] = df['minimum_requirements'].apply(lambda x: extract_cpu2(x))

In [64]:
df[df['rec_cpu'].isna()]

,app_id,name,minimum_requirements,recommended_requirements,min_cpu,rec_cpu
13,393080,Call of Duty®: Modern Warfare® Remastered,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",Intel Core i3-3225 @ 3.30GHz or equivalent,NaN
33,638970,Yakuza 0,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",Intel Core i5-3470 | AMD FX-6300,NaN
58,799640,Dungeon Munchies,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",Intel Core i5,NaN
62,364470,The Elder Scrolls®: Legends™,"Minimum:,OS:,Windows 7 / Windows 8 / Windows 1...","Recommended:,Additional Notes:,Keyboard and mouse",Intel Pentium D or AMD® Athlon™ 64 X2,NaN
65,813820,Realm Royale,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",Intel(R) Core(TM) i5-2320 CPU @ 3.00 GHz (4 CPUs),NaN
...,...,...,...,...,...,...
40741,894620,ATONE: Heart of the Elder Tree,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",Intel Core i3 3217U 1.8 GHZ Dual Core,NaN
40778,703860,GRID,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",TBC,NaN
40792,915430,Triteckka: The pure shooter,"Minimum:,OS:,Windows 10,Processor:,Dual Core C...","Recommended:,DirectX:,Version 10",Dual Core CPU,NaN
40804,909470,Touch Type Tale - Strategic Typing,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",NaN,NaN


In [66]:
df['min_cpu'].isna().sum()

np.int64(1139)